In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, f1_score
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
import seaborn as sns
import os
import random
import copy

# For reproducibility
np.random.seed(42)
random.seed(42)        # client selection
torch.manual_seed(42)  # weight init, dropout, random_split

# Device configuration (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

learning_rates = [0.005, 0.004, 0.003, 0.002, 0.001]
batch_size = 64

epochs = 3 #Number of local epochs per round (fewer local epochs + more rounds reduces client drift)
num_clients = 10  # Number of clients in federated learning
federated_rounds = 10  # Number of rounds of federated training
# as we increase the rounds the accuracy improves
selection_factor = 0.8 # Fraction of clients selected each round (more participants per round = more stable aggregation)

lr_decay_factor = 0.5  # multiply every layer's lr by this every lr_decay_every rounds
lr_decay_every = 3     # how often (in rounds) to decay; scale with federated_rounds or lr decays to ~0 early

dirichlet_alpha = 0.5  # label skew across clients: smaller = more non-IID (0.1 extreme, 100 ~ IID)

model_type = "cnn"     # "cnn" or "mlp" (the 784-128-64-10 MLP from the other notebooks)

In [ ]:
transform = transforms.ToTensor()

# Load the training dataset
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)

# Load the testing dataset
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)


In [ ]:
# Convert the datasets to numpy arrays for further processing
def convert_to_numpy(dataset):
    X = []
    y = []
    for img, label in dataset:
        X.append(img.cpu().numpy().flatten())  # Flatten the 28x28 image to a 1D array of 784 elements
        y.append(label)
    return np.array(X), np.array(y)
    # return torch.tensor(np.array(X)),torch.tensor(np.array(y))

In [ ]:
def dataset_split(dataset):
    X, y = convert_to_numpy(dataset)

    return X, y

In [ ]:
# Non-IID split: for each class, divide its samples across clients using proportions
# drawn from Dirichlet(alpha). Small alpha -> each client sees only a few classes.
def dirichlet_split(subset, num_clients, alpha, min_size=10):
    labels = torch.as_tensor(subset.dataset.targets).numpy()[subset.indices]
    num_classes = labels.max() + 1

    # Resample until every client has at least min_size samples (tiny alpha can leave a client empty)
    while True:
        client_indices = [[] for _ in range(num_clients)]
        for c in range(num_classes):
            idx_c = np.where(labels == c)[0]
            np.random.shuffle(idx_c)
            proportions = np.random.dirichlet([alpha] * num_clients)
            cuts = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
            for client_id, part in enumerate(np.split(idx_c, cuts)):
                client_indices[client_id].extend(part.tolist())
        if min(len(ci) for ci in client_indices) >= min_size:
            break

    client_datasets = []
    for client_id, ci in enumerate(client_indices):
        # Indices were appended class by class; train() batches in order, so unshuffled
        # data would give single-class batches
        np.random.shuffle(ci)
        client_datasets.append(torch.utils.data.Subset(subset, ci))
        counts = np.bincount(labels[ci], minlength=num_classes)
        print(f"Client {client_id}: {len(ci)} samples, per-class {counts.tolist()}")

    return client_datasets

In [ ]:
class Dense:
    def __init__(self, input_size, output_size, lr=0.005, beta1=0.9, beta2=0.999, epsilon=1e-8, device='cpu'):
        self.device = device
        self.weights = torch.randn(input_size, output_size, device=device) * np.sqrt(2.0 / input_size)
        self.bias = torch.zeros(1, output_size, device=device)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m_w, self.v_w = torch.zeros_like(self.weights), torch.zeros_like(self.weights)
        self.m_b, self.v_b = torch.zeros_like(self.bias), torch.zeros_like(self.bias)
        self.t = 0  # Time step

    def forward(self, X):
        self.input = X
        return torch.mm(X, self.weights) + self.bias  # Matrix multiplication (for GPU support)

    def backward(self, d_output):
        prev_weights = self.weights
        d_weights = torch.mm(self.input.T, d_output)
        d_bias = torch.sum(d_output, axis=0, keepdims=True)

        # Update Adam time step
        self.t += 1

        # Moving averages of gradients for weights
        self.m_w = self.beta1 * self.m_w + (1 - self.beta1) * d_weights
        self.v_w = self.beta2 * self.v_w + (1 - self.beta2) * (d_weights ** 2)

        # Moving averages of gradients for biases
        self.m_b = self.beta1 * self.m_b + (1 - self.beta1) * d_bias
        self.v_b = self.beta2 * self.v_b + (1 - self.beta2) * (d_bias ** 2)

        # Bias correction
        m_w_hat = self.m_w / (1 - self.beta1 ** self.t)
        v_w_hat = self.v_w / (1 - self.beta2 ** self.t)
        m_b_hat = self.m_b / (1 - self.beta1 ** self.t)
        v_b_hat = self.v_b / (1 - self.beta2 ** self.t)

        self.weights -= self.lr * m_w_hat / (torch.sqrt(v_w_hat) + self.epsilon)
        self.bias -= self.lr * m_b_hat / (torch.sqrt(v_b_hat) + self.epsilon)

        # Return the gradient for the previous layer
        return torch.mm(d_output, prev_weights.T)


class ReLU:
    def forward(self, X):
        self.input = X
        return torch.maximum(torch.tensor(0.0, device=X.device), X)

    def backward(self, d_output):
        return d_output * (self.input > 0)


In [ ]:
class Dropout:
    def __init__(self, rate=0.2):
        self.rate = rate

    def forward(self, X, training=True):
        if training:
            self.mask = (torch.rand(*X.shape, device=X.device) > self.rate) / (1 - self.rate)
            return X * self.mask
        else:
            return X

    def backward(self, d_output):
        return d_output * self.mask

In [ ]:
class BatchNorm:
    def __init__(self, input_dim, lr=0.005, epsilon=1e-5, momentum=0.9, device='cpu'):
        self.device = device
        self.lr = lr
        self.epsilon = epsilon
        self.momentum = momentum
        self.gamma = torch.ones((1, input_dim), device=device)
        self.beta = torch.zeros((1, input_dim), device=device)
        self.running_mean = torch.zeros((1, input_dim), device=device)
        self.running_var = torch.ones((1, input_dim), device=device)

    def forward(self, X, training=True):
        if training:
            batch_mean = torch.mean(X, axis=0, keepdims=True)
            batch_var = torch.var(X, axis=0, keepdims=True)

            self.X_centered = X - batch_mean
            self.stddev_inv = 1. / torch.sqrt(batch_var + self.epsilon)
            X_norm = self.X_centered * self.stddev_inv

            out = self.gamma * X_norm + self.beta  # to give the model flexibility to cancel out normalization
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * batch_mean
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * batch_var
        else:
            X_norm = (X - self.running_mean) / torch.sqrt(self.running_var + self.epsilon)
            out = self.gamma * X_norm + self.beta

        return out

    def backward(self, d_output):
        n_samples = d_output.shape[0]

        d_gamma = torch.sum(d_output * self.X_centered * self.stddev_inv, axis=0, keepdims=True)
        d_beta = torch.sum(d_output, axis=0, keepdims=True)

        d_X_norm = d_output * self.gamma
        d_var = torch.sum(d_X_norm * self.X_centered * -0.5 * self.stddev_inv**3, axis=0)
        d_mean = torch.sum(d_X_norm * -self.stddev_inv, axis=0) + d_var * torch.mean(-2. * self.X_centered, axis=0)

        d_input = d_X_norm * self.stddev_inv + d_var * 2 * self.X_centered / n_samples + d_mean / n_samples
        self.gamma -= self.lr * d_gamma
        self.beta -= self.lr * d_beta

        return d_input



In [ ]:
# Conv and MaxPool wrap PyTorch's layers but keep this network's forward/backward
# interface: autograd runs only inside the layer, the rest of the chain stays manual.

class Conv:
    def __init__(self, in_channels, out_channels, kernel_size, lr=0.005, beta1=0.9, beta2=0.999, epsilon=1e-8, device='cpu'):
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=kernel_size // 2).to(device)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        # Same hand-written Adam as Dense, one (m, v) pair per parameter (weight, bias)
        self.m = [torch.zeros_like(p) for p in self.conv.parameters()]
        self.v = [torch.zeros_like(p) for p in self.conv.parameters()]
        self.t = 0

    def forward(self, X):
        self.input = X.detach().requires_grad_(True)
        self.output = self.conv(self.input)
        return self.output.detach()

    def backward(self, d_output):
        self.output.backward(d_output)  # fills conv.weight.grad, conv.bias.grad and input.grad
        self.t += 1

        with torch.no_grad():
            for k, p in enumerate(self.conv.parameters()):
                self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * p.grad
                self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * (p.grad ** 2)
                m_hat = self.m[k] / (1 - self.beta1 ** self.t)
                v_hat = self.v[k] / (1 - self.beta2 ** self.t)
                p -= self.lr * m_hat / (torch.sqrt(v_hat) + self.epsilon)
                p.grad = None

        d_input = self.input.grad
        # Drop graph references: deepcopy (receive_model) fails on tensors that still carry a graph
        self.input, self.output = None, None
        return d_input


class MaxPool:
    def __init__(self, kernel_size=2):
        self.pool = nn.MaxPool2d(kernel_size)

    def forward(self, X):
        self.input = X.detach().requires_grad_(True)
        self.output = self.pool(self.input)
        return self.output.detach()

    def backward(self, d_output):
        self.output.backward(d_output)
        d_input = self.input.grad
        self.input, self.output = None, None
        return d_input


# Reshape between flat vectors and images; backward just undoes the reshape
class Reshape:
    def __init__(self, shape):
        self.shape = shape

    def forward(self, X):
        self.input_shape = X.shape
        return X.reshape(X.shape[0], *self.shape)

    def backward(self, d_output):
        return d_output.reshape(self.input_shape)


# Per-channel BatchNorm for conv feature maps: view (N, C, H, W) as (N*H*W, C) rows and
# reuse BatchNorm's math, so each channel gets one mean/var/gamma/beta over batch + pixels.
# As a BatchNorm subclass, Client/Server already send and average its params and running stats.
class BatchNorm2d(BatchNorm):
    def forward(self, X, training=True):
        N, C, H, W = X.shape
        out = super().forward(X.permute(0, 2, 3, 1).reshape(-1, C), training)
        return out.reshape(N, H, W, C).permute(0, 3, 1, 2)

    def backward(self, d_output):
        N, C, H, W = d_output.shape
        d_input = super().backward(d_output.permute(0, 2, 3, 1).reshape(-1, C))
        return d_input.reshape(N, H, W, C).permute(0, 3, 1, 2)

In [ ]:
class Softmax:
    def forward(self, x):
        x_max = torch.max(x, axis=1, keepdims=True).values
        exp_values = torch.exp(x - x_max)
        sum_exp_values = torch.sum(exp_values, axis=1, keepdims=True)
        epsilon = 1e-10
        probabilities = exp_values / (sum_exp_values + epsilon)
        return probabilities

    def backward(self, output, y):
        n_samples = y.shape[0]
        grad = output.clone()
        grad[range(n_samples), y] -= 1
        grad = grad / n_samples
        return grad

def cross_entropy_loss(predictions, labels):
    n_samples = labels.shape[0]
    logp = -torch.log(predictions[range(n_samples), labels] + 1e-15)
    loss = torch.sum(logp) / n_samples
    return loss


In [ ]:
class NeuralNetwork:
    def __init__(self, input_dim, num_classes, lr=0.001, device='cpu', model_type="cnn"):
        self.input_dim = input_dim
        self.num_classes= num_classes
        self.device = device
        self.softmax = Softmax()
        if model_type == "cnn":
            self.layers = [
                Reshape((1, 28, 28)),                    # flat 784 -> image
                Conv(1, 32, 3, lr, device=device),       # 32 x 28 x 28
                BatchNorm2d(32, device=device),
                ReLU(),
                MaxPool(2),                              # 32 x 14 x 14
                Conv(32, 64, 3, lr, device=device),      # 64 x 14 x 14
                BatchNorm2d(64, device=device),
                ReLU(),
                MaxPool(2),                              # 64 x 7 x 7
                Reshape((64 * 7 * 7,)),                  # image -> flat 3136
                Dense(64 * 7 * 7, 128, lr, device=device),
                BatchNorm(128, device=device),
                ReLU(),
                Dropout(),
                Dense(128, num_classes, lr, device=device)
            ]
        elif model_type == "mlp":
            self.layers = [
                Dense(input_dim, 128, lr, device=device),
                BatchNorm(128, device=device),
                ReLU(),
                Dropout(),
                Dense(128, 64, lr, device=device),
                BatchNorm(64, device=device),
                ReLU(),
                Dropout(),
                Dense(64, num_classes, lr, device=device)
            ]
        else:
            raise ValueError(f"model_type must be 'cnn' or 'mlp', got {model_type!r}")

    def forward(self, X, training=True):
        X = X.to(self.device)
        for layer in self.layers:
            if isinstance(layer, (BatchNorm, Dropout)):
                X = layer.forward(X, training=training)
            else:
                X = layer.forward(X)

        return self.softmax.forward(X)

    def backward(self, d_output):
        for layer in reversed(self.layers):
            d_output = layer.backward(d_output)

    #def train(self, X_train, y_train, X_val, y_val, epochs, batch_size):

    def train(self, X_train, y_train, epochs, batch_size):

        n_samples = X_train.shape[0]
        training_loss, val_loss, training_acc, val_acc, val_f1 = [], [], [], [], []

        for epoch in range(epochs):
            losses = []
            # disable=True: the bar re-renders on every run and churns the notebook diff
            for i in tqdm(range(0, n_samples, batch_size), disable=True):
                x_batch = torch.tensor(X_train[i:i+batch_size], dtype=torch.float32).to(self.device)
                y_batch = torch.tensor(y_train[i:i+batch_size], dtype=torch.long).to(self.device)

                # Forward pass
                output = self.forward(x_batch, training=True)

                # Loss calculation
                loss = cross_entropy_loss(output, y_batch)
                losses.append(loss)

                # Backward pass
                d_output = self.softmax.backward(output, y_batch)
                self.backward(d_output)

            # Training Loss & Accuracy
            train_preds = self.predict(X_train)
            train_accuracy = accuracy_score(y_train, train_preds)
            losses = [loss.item() if isinstance(loss, torch.Tensor) else loss for loss in losses]
            training_loss.append(np.mean(losses))
            # training_loss.append(np.mean(losses.cpu().numpy()))
            training_acc.append(train_accuracy)

            # Validation Loss & Accuracy
            # val_preds = self.predict(X_val)
            # val_accuracy = accuracy_score(y_val, val_preds)
            # val_f1_score = f1_score(y_val, val_preds, average="macro")
            # val_loss_epoch = cross_entropy_loss(self.forward(torch.tensor(X_val, dtype=torch.float32).to(self.device), training=False), torch.tensor(y_val, dtype=torch.long).to(self.device))

            # val_loss.append(val_loss_epoch.item())
            # val_acc.append(val_accuracy)
            # val_f1.append(val_f1_score)

            # print(f"Epoch {epoch+1}/{epochs}, Train Loss: {training_loss[-1]:.4f}, "
            #       f"Val Loss: {val_loss[-1]:.4f}, Train Acc: {train_accuracy:.4f}, "
            #       f"Val Acc: {val_accuracy:.4f}, Val F1: {val_f1_score:.4f}")

            print(f"Epoch {epoch+1}/{epochs}, Train Loss: {training_loss[-1]:.4f}, "
                  f"Train Acc: {train_accuracy:.4f}, ")




        # return training_loss, val_loss, training_acc, val_acc, val_f1
        return training_loss, training_acc

    def predict(self, X_test, chunk_size=1000):
        predictions = []
        # Chunked + no_grad: a conv forward over all 12k validation images at once needs GBs of activations
        with torch.no_grad():
            for i in range(0, X_test.shape[0], chunk_size):
                X_chunk = torch.tensor(X_test[i:i+chunk_size], dtype=torch.float32).to(self.device)
                output = self.forward(X_chunk, training=False)
                predictions.append(torch.argmax(output, axis=1).cpu().numpy())  # Move to CPU for numpy conversion

        # Layers cache their last activations for backward; after a 1000-image chunk that's
        # ~500MB per model, which receive_model's deepcopy would then clone into every client
        for layer in self.layers:
            for attr in ("input", "output", "mask", "X_centered", "stddev_inv"):
                if hasattr(layer, attr):
                    setattr(layer, attr, None)

        return np.concatenate(predictions)

    def evaluate(self, validation_dataset, test_dataset):

        X_val, y_val = dataset_split(validation_dataset)
        val_preds = self.predict(X_val)
        val_accuracy = accuracy_score(y_val, val_preds)
        #val_f1_score = f1_score(y_val, val_preds, average="macro")
        # val_loss_epoch = cross_entropy_loss(self.forward(torch.tensor(X_val, dtype=torch.float32).to(self.device), training=False), torch.tensor(y_val, dtype=torch.long).to(self.device))

        # val_loss.append(val_loss_epoch.item())
        # val_acc.append(val_accuracy)
        # val_f1.append(val_f1_score)

        X_test, y_test = dataset_split(test_dataset)

        test_preds = self.predict(X_test)
        test_accuracy = accuracy_score(y_test, test_preds)

        print( f"Val Acc: {val_accuracy:.4f}, Test Acc: {test_accuracy:.4f}")




## Depth: tried 3 hidden layers, no help

*Measured on the IID version (`FL_mlp_iid.ipynb`) — not re-tested on non-IID data.*

Tested 784→128→128→64→10 (one more Dense+BN+ReLU+Dropout block) vs the baseline 784→128→64→10, same FL settings (10 rounds, 3 local epochs, selection=0.8, lr decay).

Baseline: 0.8766 test acc. Deeper: 0.8737 — slightly worse.

As expected: a flat-pixel MLP plateaus in the high-80s on FashionMNIST regardless of depth, and FedAvg's neuron-misalignment across independently-trained clients only compounds with more layers. Not adopting — architecture family (CNN), not MLP depth, is the real lever.

## Aggregation: why plain weighted FedAvg, not server-side Adam

*Measured on the IID version (`FL_mlp_iid.ipynb`) — not re-tested on non-IID data, where client drift is larger and the result may differ.*

Tried a FedAdam-style server-side Adam optimizer applied to the aggregated pseudo-gradient (`aggregated_params - current_weights`) instead of directly overwriting the global model's weights with the aggregated average. Idea: give the global model persistent momentum across rounds, since each client's own Adam state resets to zero every round anyway.

Measured worse than direct replacement at every round count tested:

| Rounds | Direct replace (test acc) | Server-side Adam (test acc) |
|---|---|---|
| 1  | 0.8089 | 0.3428 |
| 10 | 0.8700 | 0.8284 |
| 40 | 0.8842 | 0.8723 |

More rounds stabilizes server-side Adam (it stops oscillating and the gap narrows), but it never catches up — Adam needs many time steps to build trustworthy statistics, which this scale of FL (tens of rounds) doesn't provide. Server-side adaptive optimizers fit cross-device FL with hundreds/thousands of rounds, not this setup. Kept plain weighted FedAvg below.

In [ ]:
class Server:

    def __init__(self ,model, validation_dataset, test_dataset, learning_rate = 0.01, device= "cuda"):
        self.global_model = model
        self.clientList = [] #List of all clients registered
        self.selectedClientList = [] #List of clients selected in a specific round
        self.markedClientList = [] #List of clients marked vulnerable( Not yet used )
        self.learning_rate = learning_rate
        self.device = device
        self.validation_dataset = validation_dataset
        self.test_dataset = test_dataset

    # Register Clients

    def register(self, clients):
        self.clientList.append(clients)
        print("Registering client")


    # Select Clients in each round

    def select(self, clientMap):

        client_id, client = clientMap
        self.selectedClientList.append(clientMap)
        print("Selecting client ", client_id)

    # Reset the server in each round

    def reset(self):
        self.selectedClientList = []

    # Aggregrate updates

    def aggregate(self, localUpdates, client_sizes):

        print("Getting aggregated by server")

        # FedAvg weighted by each client's local sample count (n_k/n), not a plain mean
        total_samples = sum(client_sizes)
        client_weights = torch.tensor([size / total_samples for size in client_sizes], device=self.device)

        aggregated_params = {}
        for key in localUpdates[0].keys():
            stacked = torch.stack([update[key].float().to(self.device) for update in localUpdates], dim=0)
            weight_shape = [len(client_sizes)] + [1] * (stacked.dim() - 1)
            aggregated_params[key] = (stacked * client_weights.view(weight_shape)).sum(dim=0)

        return aggregated_params


    # Update global model with aggregrated parameters


    def update_global_model(self, localUpdates, client_sizes):

        aggregated_params = self.aggregate(localUpdates, client_sizes)

        # Layer by layer update weight to model self.global_model.
        for i, layer in enumerate(self.global_model.layers):
            if isinstance(layer, Conv):
                # copy_ in place: the nn.Conv2d must keep its own Parameter objects
                layer.conv.weight.data.copy_(aggregated_params[f"layer_{i}_weights"])
                layer.conv.bias.data.copy_(aggregated_params[f"layer_{i}_bias"])
            elif isinstance(layer, Dense):
                layer.weights = aggregated_params[f"layer_{i}_weights"]
                layer.bias = aggregated_params[f"layer_{i}_bias"]
            elif isinstance(layer, BatchNorm):
                layer.gamma = aggregated_params[f"layer_{i}_gamma"]
                layer.beta = aggregated_params[f"layer_{i}_beta"]
                layer.running_mean = aggregated_params[f"layer_{i}_running_mean"]
                layer.running_var = aggregated_params[f"layer_{i}_running_var"]


        print("Done Aggregating")

    # Decay every layer's own learning rate (Conv/Dense Adam lr, BatchNorm's SGD lr)
    def decay_learning_rate(self, factor):
        for layer in self.global_model.layers:
            if isinstance(layer, (Conv, Dense, BatchNorm)):
                layer.lr *= factor
        dense_lr = next(layer.lr for layer in self.global_model.layers if isinstance(layer, Dense))
        print(f"Decayed learning rate by {factor}, Dense lr now {dense_lr:.6f}")

    def evaluate(self):
      self.global_model.evaluate(self.validation_dataset, self.test_dataset)

    # Distribute updated model to clients

    def distribute_global_model(self):
        for client in self.clientList:
            client.receive_model(self.global_model)

In [ ]:

# Federated Learning Client Class
class Client:
    def __init__(self, model, client_train_data, epochs, batch_size, device='cpu'):
        self.client_train_data=client_train_data
        self.num_samples = len(client_train_data)
        #self.client_test_data=client_test_data
        # Private copy: sharing one model object makes every client train the same tensors
        self.model = copy.deepcopy(model)
        self.device = device


    def train(self):

        X_train, y_train = dataset_split(self.client_train_data)
         # print(f'Train Data Shape: {X_train.shape}, Train Labels Shape: {y_train.shape}')
            # print(f'Train Data Shape: {X_val.shape}, Train Labels Shape: {y_val.shape}')
            # print(f'Test Data Shape: {X_test.shape}, Test Labels Shape: {y_test.shape}')
            # print(y_test[:10])

        # train_loss, val_loss, train_acc, val_acc, val_f1 = self.model.train(
        #     X_train, y_train, X_val, y_val, epochs, batch_size
        # )

        # return train_loss, val_loss, train_acc, val_acc, val_f1

        #train_loss, train_acc = self.model.train( X_train, y_train, X_val, y_val, epochs, batch_size )

        train_loss, train_acc = self.model.train( X_train, y_train, epochs, batch_size )

    def get_parameters(self):
        weights = {}
        # clone(): layers update weights in-place, so an un-cloned snapshot would keep changing
        for i, layer in enumerate(self.model.layers):
            if isinstance(layer, Conv):
                weights[f"layer_{i}_weights"] = layer.conv.weight.detach().clone()
                weights[f"layer_{i}_bias"] = layer.conv.bias.detach().clone()
            elif isinstance(layer, Dense):
                weights[f"layer_{i}_weights"] = layer.weights.clone()
                weights[f"layer_{i}_bias"] = layer.bias.clone()
            elif isinstance(layer, BatchNorm):
                weights[f"layer_{i}_gamma"] = layer.gamma.clone()
                weights[f"layer_{i}_beta"] = layer.beta.clone()
                weights[f"layer_{i}_running_mean"] = layer.running_mean.clone()
                weights[f"layer_{i}_running_var"] = layer.running_var.clone()


        return weights

    def receive_model(self,global_model):
        self.model=copy.deepcopy(global_model)



In [ ]:

# Main Federated Learning Process
def federated_learning_simulation(epochs, batch_size, learning_rates, num_clients, federated_rounds, selection_factor, train_dataset, test_dataset, lr_decay_factor, lr_decay_every, dirichlet_alpha, model_type ):

    # X_train, X_val, y_train, y_val, X_test, y_test = format_data(train_dataset,test_dataset)


    # # Optionally, if you want to divide the test dataset as well (e.g., each client gets a portion of the test set)
    #client_test_data = torch.utils.data.random_split(test_dataset, [len(test_dataset) // num_clients] * num_clients)

    #client_train_data = torch.utils.data.random_split(train_dataset, [len(train_dataset) // num_clients] * num_clients)
    # X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    # y_train_tensor = torch.tensor(y_train, dtype=torch.int)



    # train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)

    # Hold the validation split out first, so no client ever trains on it
    val_size = len(train_dataset) // 5
    train_subset, val_subset = torch.utils.data.random_split(train_dataset, [len(train_dataset) - val_size, val_size])

    # Validation/test stay IID (global class balance); only the client training shards are skewed
    client_train_data = dirichlet_split(train_subset, num_clients, dirichlet_alpha)

    # X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    # y_val_tensor = torch.tensor(y_val, dtype=torch.int)

    # validation_dataset = torch.utils.data.TensorDataset(X_val_tensor, y_val_tensor)

    global_model = NeuralNetwork(input_dim=784, num_classes=10, lr=learning_rates[0], device=device, model_type=model_type)

    server = Server(model=global_model, learning_rate = 0.01, validation_dataset = val_subset, test_dataset = test_dataset, device=device)

    for i in range(0,num_clients):
        client = Client(global_model, client_train_data[i], epochs, batch_size, device)
        server.register(client)


    for round in range(federated_rounds):

        server.reset()

        for i in range(0,num_clients):

            rd = random.randint(0,100)
            if rd < selection_factor*100:
                clientMap = i, server.clientList[i]
                server.select( clientMap )




        client_weights=[]
        selected_client_sizes=[]

        for client_id,client in server.selectedClientList:

            print("Training started by client ", client_id )

            client.train()

            client_weights.append(client.get_parameters())
            selected_client_sizes.append(client.num_samples)

            print("Training completed by client ", client_id)

        server.update_global_model(client_weights, selected_client_sizes)
        server.evaluate()

        if (round + 1) % lr_decay_every == 0:
            server.decay_learning_rate(lr_decay_factor)

        server.distribute_global_model()



federated_learning_simulation(epochs, batch_size, learning_rates, num_clients, federated_rounds, selection_factor, train_dataset, test_dataset, lr_decay_factor, lr_decay_every, dirichlet_alpha, model_type )

In [ ]:


# # Make predictions with the best model
# Y_pred = best_model.predict(X_test)
# accuracy = accuracy_score(y_test, Y_pred)
# precision = precision_score(y_test, Y_pred, average="weighted")
# recall = recall_score(y_test, Y_pred, average="weighted")
# f1 = f1_score(y_test, Y_pred, average="weighted")

# # Print the evaluation metrics for the best model
# print(f"Accuracy: {accuracy}")
# print(f"Precision: {precision}, Recall: {recall}, F1 Score: {f1}")

